# 02 — Train the career retriever

You are **fine-tuning a small sentence embedding model** (`all-MiniLM-L6-v2`) on career question–passage pairs.

That is the model your supervisor means. You are **not** training GPT.

- Loss: Multiple Negatives Ranking (pull a question toward its true passage)
- Hardware: CPU is enough for this tiny dataset
- Output: `ml/models/career-retriever`

First run costs time while the base MiniLM downloads (~80MB).

In [1]:
from pathlib import Path
import sys
import pandas as pd
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, InputExample, losses

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.append(str(ROOT / "src"))
from config import QA, TRAINED_DIR

df = pd.read_csv(QA)
train = df[df["split"] == "train"]
print(train.shape)
train.head()

C:\Users\Mwiti\Downloads\AICareerCoach\ml\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(23, 4)


C:\Users\Mwiti\AppData\Local\Temp\ipykernel_39752\3999678063.py:5: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, InputExample, losses


,split,query,resource_id,positive_passage
0,train,How should a computer science student structur...,software_engineer_cv,A strong internship or graduate CV for compute...
1,train,What belongs in CV bullets?,software_engineer_cv,Write bullets as action + tool + result. Examp...
2,train,Should I put a photo on my CV?,software_engineer_cv,Include a GitHub or portfolio link. Do not att...
3,train,How do I match my CV to a job posting?,software_engineer_cv,"If the posting asks for Git, SQL, and teamwork..."
4,train,How long should a cover letter be?,cover_letter_guide,A cover letter should be under 300 words. Para...


In [2]:
examples = [
    InputExample(texts=[row.query, row.positive_passage])
    for row in train.itertuples(index=False)
]
loader = DataLoader(examples, shuffle=True, batch_size=4)

model = SentenceTransformer("all-MiniLM-L6-v2")
train_loss = losses.MultipleNegativesRankingLoss(model)

TRAINED_DIR.mkdir(parents=True, exist_ok=True)
model.fit(
    train_objectives=[(loader, train_loss)],
    epochs=4,
    warmup_steps=5,
    show_progress_bar=True,
    output_path=str(TRAINED_DIR),
)
print("Saved trained retriever to", TRAINED_DIR)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3371.10it/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

C:\Users\Mwiti\Downloads\AICareerCoach\ml\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  8.04it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  7.89it/s]

Saved trained retriever to C:\Users\Mwiti\Downloads\AICareerCoach\ml\models\career-retriever
